# Phase 2 — Batch Test & Patches

Self-contained. Folds in both outstanding patches and runs the pipeline across several videos.

### Patches applied

**Speed.** The activity gate did a random seek per sampled frame — on a multi-GB h264 file that means thousands of decoder resets. Replaced with `grab()`/`retrieve()`: sequential decode, converting only sampled frames. Measured **9.2× faster** on the decode.

**Per-class abstention.** A single global floor cannot serve a model whose classes range from 0.974 precision (serve) to 0.403 (defence). One threshold either wastes reliable serves or admits defence labels that are wrong 6 times in 10. Now one threshold per class, fitted on held-out predictions.

### What this test can and cannot tell you

The final models trained on all 12 videos except `test_5`. **There is no held-out video**, so this is a *pipeline* test, not an accuracy test:

- does it run without crashing on every video
- are shot counts within ~10% of ground truth
- does it hold up in the second venue (`game_5`, `test_7` — green wall, red floor)
- what is the real runtime after the speed patch
- how much gets abstained per video

`test_5` is included deliberately as a **negative control**. Its pose stream carries no contact signal (P 0.008 at labelled frames vs 0.006 elsewhere), and the models never saw it. It should find few or no shots. If it finds many, something is wrong with the test, not with test_5.


## 1 · Mount & install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."
print(f"GPU: {torch.cuda.get_device_name(0)}")

# rtmlib declares CPU onnxruntime as a hard dependency; install it --no-deps
# and put the GPU wheel LAST or RTMPose silently drops to CPU.
!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q opencv-python numpy tqdm ultralytics pyarrow 2>&1 | tail -1
!pip install -q "onnxruntime-gpu==1.22.0" 2>&1 | tail -1
!pip list 2>/dev/null | grep -iE "onnxruntime|rtmlib|ultralytics"

import os, glob, site
libs = []
for sp in site.getsitepackages():
    libs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
if libs: open("/content/_ort_libpath.txt","w").write(":".join(libs))
print("\n" + "="*56)
print("  NOW: Runtime > Restart session, then run from cell 3.")
print("="*56)

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.1 MB/s eta 0:00:00
onnxruntime-gpu                       1.22.0
rtmlib                                0.0.16
ultralytics                           8.4.128
ultralytics-platform                  0.1.14
ultralytics-thop                      2.1.6

  NOW: Runtime > Restart session, then run from cell 3.


## 2 · Config

In [3]:
BASE = "/content/drive/MyDrive/tt_coach"

# Short videos first — the full set would take hours. game_1 is already done.
TEST_VIDEOS = ["test_2", "test_6", "test_1", "test_7", "test_5"]
#              0.5min    1.5min    2.3min   2.2min   1.6min  (~8 min of video)
# test_7 is the SECOND VENUE (green wall, red floor) — the venue check.
# test_5 is the NEGATIVE CONTROL — never trained on, no contact signal.

ACT_STRIDE  = 12
ACT_PAD_S   = 0.75
MOTION_THR  = 0.02
DET_THR     = 0.60
NMS_GAP     = 30
RALLY_GAP_S = 2.5

import json, math, shutil, time, os, glob, site
from pathlib import Path
import numpy as np, pandas as pd

_lp = Path("/content/_ort_libpath.txt")
if _lp.exists():
    os.environ["LD_LIBRARY_PATH"] = _lp.read_text()+":"+os.environ.get("LD_LIBRARY_PATH","")

import cv2, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

BASE = Path(BASE); META = BASE/"derived/meta"; CKPT = BASE/"models/checkpoints"
ANALYSED = BASE/"derived/analysed"; ANALYSED.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/_work"); LOCAL.mkdir(exist_ok=True)
dev = "cuda"

folds = json.loads((META/"folds.json").read_text())
PRE, NF, FPS = folds["window"]["pre"], folds["window"]["n_frames"], 120
CLASSES = ["serve","attack","control","defence"]
TECHS = ["block","chop","flick","lob","loop","push","serve","smash"]
L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI = 5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK = 11,12,13,14,15,16
FLIP = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

CAL = json.loads((META/"calibration.json").read_text())
TEMP = CAL["temperature"]
THRESHOLDS = CAL.get("per_class_thresholds",
                     {"serve":0.25,"attack":0.50,"control":1.01,"defence":1.01})
print(f"temperature {TEMP:.2f}")
print(f"thresholds  { {k: round(v,2) for k,v in THRESHOLDS.items()} }")
sup = [k for k,v in THRESHOLDS.items() if v > 1.0]
print(f"suppressed  {sup}  (counted in rallies, never coached)")

def load(stem):
    p = META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")
strokes = load("strokes")

temperature 2.20
thresholds  {'serve': 0.25, 'attack': 0.5, 'control': 1.01, 'defence': 1.01}
suppressed  ['control', 'defence']  (counted in rallies, never coached)


## 3 · Models

In [4]:
class Block(nn.Module):
    def __init__(s,c,d,drop=0.1):
        super().__init__()
        s.c1=nn.Conv1d(c,c,5,padding=2*d,dilation=d); s.c2=nn.Conv1d(c,c,5,padding=2*d,dilation=d)
        s.n1,s.n2=nn.BatchNorm1d(c),nn.BatchNorm1d(c); s.do=nn.Dropout(drop)
    def forward(s,x):
        r=x; x=s.do(F.gelu(s.n1(s.c1(x)))); x=s.do(F.gelu(s.n2(s.c2(x)))); return F.gelu(x+r)

class DetNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d) for d in (1,2,4,8,16,32,64)])
        s.hc,s.hs=nn.Conv1d(w,1,1),nn.Conv1d(w,1,1)
    def forward(s,x):
        z=s.blocks(s.stem(x)); return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s,c):
        super().__init__(); s.score=nn.Conv1d(c,1,1)
    def forward(s,x):
        w=torch.softmax(s.score(x),-1); return torch.cat([(x*w).sum(-1),x.max(-1).values],-1)

class ClsNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d,0.2) for d in (1,2,4,8,16,32)])
        s.pool=AttnPool(w)
        s.trunk=nn.Sequential(nn.Linear(w*2,256),nn.GELU(),nn.Dropout(0.3))
        s.shot,s.tech=nn.Linear(256,4),nn.Linear(256,8)
    def forward(s,x):
        z=s.trunk(s.pool(s.blocks(s.stem(x)))); return s.shot(z), s.tech(z)

dck=torch.load(CKPT/"detector_final.pt",map_location=dev,weights_only=False)
cck=torch.load(CKPT/"classifier_final.pt",map_location=dev,weights_only=False)
DET=DetNet(dck["c_in"]).to(dev); DET.load_state_dict(dck["state"]); DET.eval()
CLS=ClsNet(cck["c_in"]).to(dev); CLS.load_state_dict(cck["state"]); CLS.eval()
DMU,DSD=dck["mu"],dck["sd"]; CMU,CSD=cck["mu"].to(dev),cck["sd"].to(dev)

import onnxruntime as ort
assert "CUDAExecutionProvider" in ort.get_available_providers(), \
    "CUDA provider missing — restart the session after cell 1b."
from ultralytics import YOLO
from rtmlib import RTMPose
YOLO_DET=YOLO(str(BASE/"models/detector/best.pt")); YOLO_DET.to("cuda")
PLAYER_CLS=[k for k,v in YOLO_DET.names.items() if v.lower()=="player"][0]
TABLE_CLS =[k for k,v in YOLO_DET.names.items() if v.lower()=="table"][0]
POSE=RTMPose(onnx_model=("https://download.openmmlab.com/mmpose/v1/projects/"
     "rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
     "3f5a1437_20230504.zip"),model_input_size=(288,384),
     backend="onnxruntime",device="cuda")
print(f"detector LOVO F1 {dck['lovo_f1_at_tol8']} | classifier {cck['lovo_macro_f1']}")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip
100%|██████████| 98.9M/98.9M [00:03<00:00, 32.0MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
detector LOVO F1 0.881 | classifier 0.792


## 4 · Pipeline (patched)

The activity gate now decodes **sequentially**. `grab()` advances the decoder without converting to BGR; `retrieve()` converts only the sampled frames. That avoids the keyframe reset that a random seek forces.

In [5]:
def resolve(frames, mid_x, conf=0.35):
    out=[]
    for r in YOLO_DET.predict(frames,verbose=False,conf=conf):
        d={"left":None,"right":None}
        if r.boxes is not None and len(r.boxes):
            xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
            pl=xy[cl==PLAYER_CLS]
            if len(pl):
                cx=(pl[:,0]+pl[:,2])/2; ls,rs=pl[cx<mid_x],pl[cx>=mid_x]
                if len(ls): d["left"]=ls[np.argmin((ls[:,0]+ls[:,2])/2)]
                if len(rs): d["right"]=rs[np.argmax((rs[:,0]+rs[:,2])/2)]
        out.append(d)
    return out

def find_table(cap,n,k=9):
    bx=[]
    for f in np.linspace(n*0.1,n*0.9,k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,fr=cap.read()
        if not ok: continue
        r=YOLO_DET.predict(fr,verbose=False,conf=0.35)[0]
        if r.boxes is None or not len(r.boxes): continue
        xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
        tb=xy[cl==TABLE_CLS]
        if len(tb): bx.append(tb[np.argmax((tb[:,2]-tb[:,0])*(tb[:,3]-tb[:,1]))])
    return np.median(np.stack(bx),0).astype(np.float32) if bx else None

def activity_gate(path,nfr,mid_x,stride=ACT_STRIDE):
    """PATCHED: sequential decode instead of a seek per sampled frame."""
    cap=cv2.VideoCapture(str(path)); flags,idxs,buf,bidx,prev=[],[],[],[],None
    pb=tqdm(total=nfr,desc="  gate",leave=False); i=0
    def flush():
        nonlocal buf,bidx,prev
        for j,d in enumerate(resolve(buf,mid_x)):
            both=d["left"] is not None and d["right"] is not None
            mov=True
            if both and prev is not None:
                h=max(d["left"][3]-d["left"][1],1)
                mv=max(abs((d["left"][0]+d["left"][2])/2-prev[0]),
                       abs((d["right"][0]+d["right"][2])/2-prev[1]))/h
                mov=mv>MOTION_THR
            if both: prev=((d["left"][0]+d["left"][2])/2,(d["right"][0]+d["right"][2])/2)
            flags.append(both and mov); idxs.append(bidx[j])
        buf,bidx=[],[]
    while i<nfr:
        if not cap.grab(): break
        if i%stride==0:
            ok,fr=cap.retrieve()
            if ok: buf.append(fr); bidx.append(i)
            if len(buf)>=64: flush()
        i+=1
        if i%2000==0: pb.update(2000)
    if buf: flush()
    pb.close(); cap.release()
    pad=int(ACT_PAD_S*FPS); spans=[]
    for k,f in enumerate(flags):
        if not f: continue
        s,e=max(0,idxs[k]-pad),min(nfr-1,idxs[k]+pad)
        if spans and s<=spans[-1][1]+1: spans[-1][1]=max(spans[-1][1],e)
        else: spans.append([s,e])
    return spans

def extract_pose(path,spans,mid_x):
    total=sum(e-s+1 for s,e in spans)
    F_=np.zeros(total,np.int32); KP=np.zeros((total,2,17,2),np.float16)
    SC=np.zeros((total,2,17),np.float16); BX=np.zeros((total,2,4),np.float16)
    DT=np.zeros((total,2),bool); SG=np.zeros(total,np.int32)
    cap=cv2.VideoCapture(str(path)); w=0
    pb=tqdm(total=total,desc="  pose",leave=False)
    for si,(s0,e0) in enumerate(spans):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(s0)); pos=s0
        while pos<=e0:
            n=min(600,e0-pos+1); frames=[]
            for _ in range(n):
                ok,fr=cap.read()
                frames.append(fr if ok else (frames[-1] if frames else np.zeros((720,1280,3),np.uint8)))
            n=len(frames)
            di=list(range(0,n,8)); di+=[] if di[-1]==n-1 else [n-1]
            dets=resolve([frames[i] for i in di],mid_x)
            boxes={}
            for pi,side in enumerate(["left","right"]):
                kn=[(i,b) for i,b in zip(di,[d[side] for d in dets]) if b is not None]
                if not kn: continue
                ki=np.array([a for a,_ in kn],float); kb=np.stack([b for _,b in kn]).astype(float)
                boxes[pi]=np.stack([np.interp(np.arange(n),ki,kb[:,c]) for c in range(4)],1).astype(np.float32)
            for k in range(n):
                bb,who=[],[]
                for pi in (0,1):
                    if pi in boxes:
                        b=boxes[pi][k]; bw,bh=b[2]-b[0],b[3]-b[1]
                        b=np.array([max(0,b[0]-bw*.18),max(0,b[1]-bh*.11),b[2]+bw*.18,b[3]+bh*.045],np.float32)
                        bb.append(b); who.append(pi); BX[w+k,pi]=b; DT[w+k,pi]=True
                if bb:
                    kp,sc=POSE(frames[k],bboxes=np.stack(bb))
                    for j,pi in enumerate(who): KP[w+k,pi]=kp[j]; SC[w+k,pi]=sc[j]
                F_[w+k]=pos+k; SG[w+k]=si
            w+=n; pos+=n; pb.update(n); del frames
    pb.close(); cap.release()
    return dict(frame_idx=F_[:w],seg_id=SG[:w],keypoints=KP[:w],
                scores=SC[:w],boxes=BX[:w],detected=DT[:w])
print("gate + extract ready")

gate + extract ready


## 5 · Canonicalise, decode, measure

In [6]:
def canon(kp,sc,seg,mirror):
    kp=kp.astype(np.float32).copy()
    hip=(kp[:,L_HIP]+kp[:,R_HIP])/2; sho=(kp[:,L_SHO]+kp[:,R_SHO])/2
    torso=np.linalg.norm(sho-hip,axis=-1); scale=np.ones(len(kp),np.float32)
    for s in np.unique(seg):
        m=seg==s; t=torso[m]; t=t[t>1]; scale[m]=np.median(t) if len(t) else 1.
    kp=(kp-hip[:,None,:])/np.maximum(scale,1e-3)[:,None,None]
    if mirror:
        kp[...,0]*=-1; sc=sc.copy()
        for a,b in FLIP: kp[:,[a,b]]=kp[:,[b,a]]; sc[:,[a,b]]=sc[:,[b,a]]
    return kp,sc,scale

def build_stream(raw,table):
    seg=raw["seg_id"]; KP=raw["keypoints"]; SC=raw["scores"]; DT=raw["detected"]
    ch,kps,vals,tds=[],[],[],[]
    for pi in (0,1):
        kp,sc,scale=canon(KP[:,pi],SC[:,pi].astype(np.float32),seg,mirror=(pi==1))
        vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
        vel[np.diff(seg,prepend=seg[0])!=0]=0
        ch+=[kp.reshape(len(kp),-1),vel.reshape(len(kp),-1),(sc*DT[:,pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc>=.35)&DT[:,pi:pi+1])
        hx=(KP[:,pi,L_HIP,0]+KP[:,pi,R_HIP,0])/2
        edge=table[0] if pi==0 else table[2]
        tds.append(np.abs(hx-edge)/np.maximum(scale,1e-3) if table is not None and table[2]>table[0]
                   else np.zeros(len(kp),np.float32))
    cuts=np.where(np.diff(seg)!=0)[0]+1; b=np.concatenate([[0],cuts,[len(seg)]])
    return dict(X=np.nan_to_num(np.concatenate(ch,1).astype(np.float32)),
                kp=np.stack(kps,1),val=np.stack(vals,1),td=np.nan_to_num(np.stack(tds,1)),
                fidx=raw["frame_idx"],scores=SC,detected=DT,
                spans=[(int(b[i]),int(b[i+1])) for i in range(len(b)-1)])

def decode(prob,thr=DET_THR,gap=NMS_GAP):
    idx=np.where(prob>=thr)[0]
    if not len(idx): return np.array([],int)
    pk=[i for i in idx if prob[i]==prob[max(0,i-gap//2):i+gap//2+1].max()]
    pk=sorted(pk,key=lambda i:-prob[i]); keep=[]
    for p in pk:
        if all(abs(p-k)>=gap for k in keep): keep.append(p)
    return np.array(sorted(keep),int)

def window_at(st,idx,side):
    sel=np.clip(np.arange(idx-PRE,idx-PRE+NF),0,len(st["X"])-1)
    pi=0 if side=="left" else 1
    kp=st["kp"][sel,pi]; val=st["val"][sel,pi].astype(np.float32)
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    x=np.concatenate([kp.reshape(NF,-1),vel.reshape(NF,-1),val,st["td"][sel,pi][:,None]],1)
    return np.nan_to_num(x).T.astype(np.float32),kp,val,sel

def angle(a,b,c):
    v1,v2=a-b,c-b
    cs=(v1*v2).sum(-1)/np.maximum(np.linalg.norm(v1,axis=-1)*np.linalg.norm(v2,axis=-1),1e-6)
    return np.degrees(np.arccos(np.clip(cs,-1,1)))

def kinematics(kp,val,td_win,wri):
    h=-kp[...,1]; vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    spd=np.linalg.norm(vel,axis=-1); spd[~val.astype(bool)]=np.nan
    w=spd[:,wri]; pre,post=slice(0,PRE),slice(PRE,NF)
    sh,el=(R_SHO,R_ELB) if wri==R_WRI else (L_SHO,L_ELB)
    d_hip=np.linalg.norm(kp[:,wri],axis=-1); ea=angle(kp[:,sh],kp[:,el],kp[:,wri])
    pk=np.nanmax(w) if np.isfinite(w).any() else np.nan
    rec=np.nan
    if np.isfinite(pk) and pk>0:
        a=np.where(np.nan_to_num(w[post])<0.2*pk)[0]; rec=float(a[0]) if len(a) else np.nan
    tr=(kp[:,L_SHO]+kp[:,R_SHO])/2; ta=np.degrees(np.arctan2(tr[:,0],-tr[:,1]))
    return {"backswing_amplitude":float(np.nanmax(d_hip[pre])),
        "peak_wrist_speed":float(pk),
        "time_to_peak":int(np.nanargmax(w)-PRE) if np.isfinite(w).any() else None,
        "contact_height":float(h[PRE,wri]-(h[PRE,L_SHO]+h[PRE,R_SHO])/2),
        "elbow_angle":float(ea[PRE]),"elbow_range":float(np.nanmax(ea)-np.nanmin(ea)),
        "trunk_lean":float(ta[PRE]),"trunk_rotation":float(np.nanmax(ta)-np.nanmin(ta)),
        "table_distance":float(td_win[PRE]),
        "stance_width":float(abs(kp[PRE,L_ANK,0]-kp[PRE,R_ANK,0])),
        "knee_angle":float((angle(kp[PRE:PRE+1,L_HIP],kp[PRE:PRE+1,L_KNE],kp[PRE:PRE+1,L_ANK])[0]+
                            angle(kp[PRE:PRE+1,R_HIP],kp[PRE:PRE+1,R_KNE],kp[PRE:PRE+1,R_ANK])[0])/2),
        "follow_through":float(np.nansum(w[post])),"recovery_time":rec}
print("ready")

ready


## 6 · `analyse()` — with per-class abstention

In [7]:
def analyse(video_path,video_id=None,force=False):
    video_path=Path(video_path); vid=video_id or video_path.stem
    outdir=ANALYSED/vid; outdir.mkdir(parents=True,exist_ok=True)
    cache=outdir/"pose_raw.npz"; t0=time.time()

    local=LOCAL/video_path.name
    if not local.exists(): shutil.copy(video_path,local)
    cap=cv2.VideoCapture(str(local))
    nfr=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS)
    W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); table=find_table(cap,nfr); cap.release()
    mid_x=(table[0]+table[2])/2 if table is not None else W/2
    dur=nfr/max(fps,1)
    print(f"{vid}: {nfr:,} frames, {dur/60:.1f} min")

    if cache.exists() and not force:
        raw={k:v for k,v in np.load(cache,allow_pickle=True).items()}
        raw.pop("table_box",None)
        print(f"  cached pose {len(raw['frame_idx']):,} frames")
    else:
        ta=time.time(); spans=activity_gate(local,nfr,mid_x)
        cov=sum(e-s+1 for s,e in spans)/max(nfr,1)
        print(f"  gate {time.time()-ta:.0f}s -> {len(spans)} regions, {cov:.0%}")
        raw=extract_pose(local,spans,mid_x)
        np.savez_compressed(cache,**raw,
            table_box=table if table is not None else np.zeros(4,np.float32))
    st=build_stream(raw,table)

    prob=np.zeros(len(st["X"]),np.float32); sidep=np.zeros(len(st["X"]),np.float32)
    with torch.no_grad():
        for a,b in st["spans"]:
            x=torch.tensor(((st["X"][a:b]-DMU)/DSD).T[None],dtype=torch.float32,device=dev)
            lc,ls=DET(x)
            prob[a:b]=torch.sigmoid(lc)[0].cpu().numpy()
            sidep[a:b]=torch.sigmoid(ls)[0].cpu().numpy()
    peaks=decode(prob)
    if not len(peaks):
        print("  no contacts found"); return pd.DataFrame(),None
    sides=["right" if sidep[i]>=0.5 else "left" for i in peaks]
    wins,kps,vals,sels=[],[],[],[]
    for i,s in zip(peaks,sides):
        x,kp,val,sel=window_at(st,i,s); wins.append(x);kps.append(kp);vals.append(val);sels.append(sel)
    with torch.no_grad():
        lo,lt=CLS((torch.tensor(np.stack(wins),device=dev)-CMU)/CSD)
        pr=torch.softmax(lo/TEMP,1).cpu().numpy(); pt=torch.softmax(lt/TEMP,1).cpu().numpy()

    frames=st["fidx"][peaks]
    rid,sidx,cur,last=[],[],0,None
    for f in frames:
        if last is not None and (f-last)/FPS>RALLY_GAP_S: cur+=1; k=0
        else: k=0 if last is None else sidx[-1]+1
        rid.append(cur); sidx.append(k); last=f
    rlen=pd.Series(rid).value_counts().to_dict()

    rows=[]
    for n,(i,s) in enumerate(zip(peaks,sides)):
        pi=0 if s=="left" else 1
        cl=CLASSES[int(pr[n].argmax())]
        km=kinematics(kps[n],vals[n],st["td"][sels[n],pi],R_WRI if s=="left" else L_WRI)
        rows.append(dict(video_id=vid,rally_id=rid[n],shot_index=sidx[n],player=s,
            frame=int(frames[n]),timestamp_s=float(frames[n]/FPS),shot_class=cl,
            class_confidence=float(pr[n].max()),
            technique=TECHS[int(pt[n].argmax())],
            abstain=bool(pr[n].max()<THRESHOLDS[cl]),     # PATCH: per-class
            detect_confidence=float(prob[i]),rally_length=int(rlen[rid[n]]),
            pose_confidence=float(st["scores"][sels[n],pi][st["scores"][sels[n],pi]>0].mean()
                                  if (st["scores"][sels[n],pi]>0).any() else 0.),
            detected=float(st["detected"][sels[n],pi].mean()),**km))
    df=pd.DataFrame(rows)
    df.to_parquet(outdir/"shots.parquet",index=False)
    np.savez_compressed(outdir/"pose_windows.npz",
        stroke_id=df.apply(lambda r:f"{vid}_{r.frame:07d}",axis=1).values.astype(str),
        pose_window=np.stack(kps).astype(np.float16),valid=np.stack(vals))
    el=time.time()-t0
    print(f"  {len(df)} shots, {df.rally_id.nunique()} rallies, "
          f"{df.abstain.sum()} abstained ({df.abstain.mean():.0%})  "
          f"[{el:.0f}s = {el/max(dur,1):.1f}x realtime]")
    return df,el/max(dur,1)
print("analyse() ready")

analyse() ready


## 7 · Batch run

In [8]:
results,frames_all=[],[]
for vid in TEST_VIDEOS:
    p=BASE/f"raw/videos/{vid}.mp4"
    if not p.exists(): print(f"{vid}: missing"); continue
    print("-"*60)
    try:
        df,rt=analyse(p,video_id=vid,force=True)
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        results.append(dict(video_id=vid,status="ERROR")); continue
    gt=strokes[strokes.video_id==vid]
    if df is None or not len(df):
        results.append(dict(video_id=vid,status="no shots",detected=0,
                            truth=len(gt),realtime=rt)); continue
    frames_all.append(df)
    results.append(dict(video_id=vid,status="ok",detected=len(df),truth=len(gt),
        delta=len(df)/max(len(gt),1)-1,rallies=df.rally_id.nunique(),
        mean_rally=df.groupby("rally_id").size().mean(),
        abstained=df.abstain.mean(),coachable=1-df.abstain.mean(),
        pose_conf=df.pose_confidence.mean(),realtime=rt))
res=pd.DataFrame(results)
print("\n"+"="*78); print("BATCH RESULTS"); print("="*78)
print(res.to_string(index=False,float_format=lambda x:f"{x:.3f}"))

------------------------------------------------------------
test_2: 3,600 frames, 0.5 min


  gate:   0%|          | 0/3600 [00:00<?, ?it/s]

  gate 52s -> 4 regions, 93%


  pose:   0%|          | 0/3363 [00:00<?, ?it/s]

  30 shots, 3 rallies, 3 abstained (10%)  [259s = 8.6x realtime]
------------------------------------------------------------
test_6: 10,800 frames, 1.5 min


...

  50 shots, 8 rallies, 17 abstained (34%)  [660s = 5.1x realtime]
------------------------------------------------------------
test_5: 11,520 frames, 1.6 min


  gate:   0%|          | 0/11520 [00:00<?, ?it/s]

  gate 156s -> 10 regions, 58%


  pose:   0%|          | 0/6687 [00:00<?, ?it/s]

  1 shots, 1 rallies, 1 abstained (100%)  [549s = 5.7x realtime]

BATCH RESULTS
video_id status  detected  truth  delta  rallies  mean_rally  abstained  coachable  pose_conf  realtime
  test_2     ok        30     29  0.034        3      10.000      0.100      0.900      0.835     8.637
  test_6     ok        37     39 -0.051        8       4.625      0.351      0.649      0.803     6.518
  test_1     ok        82     84 -0.024       10       8.200      0.402      0.598      0.818     6.146
  test_7     ok        50     49  0.020        8       6.250      0.340      0.660      0.803     5.081
  test_5     ok         1     27 -0.963        1       1.000      1.000      0.000      0.778     5.718


## 8 · Read the result

In [9]:
ok=res[res.status=="ok"] if "status" in res else res
print("="*78)
if len(ok):
    print(f"  videos run     : {len(ok)}/{len(TEST_VIDEOS)}")
    print(f"  shot count err : mean {ok.delta.abs().mean():+.1%}, "
          f"worst {ok.delta.abs().max():+.1%}")
    print(f"  runtime        : {ok.realtime.mean():.1f}x realtime "
          f"(was 6.6x before the speed patch)")
    print(f"  coachable      : {ok.coachable.mean():.0%} of shots")
    print(f"  mean rally     : {ok.mean_rally.mean():.1f} shots")

    within=(ok.delta.abs()<=0.10).mean()
    print(f"\n  within +/-10% of ground truth: {within:.0%} of videos"
          f"   {'PASS' if within>=0.75 else 'CHECK'}")

    # venue check — test_7 is the green-wall/red-floor venue
    if "test_7" in set(ok.video_id):
        v7=ok[ok.video_id=="test_7"].iloc[0]
        others=ok[(ok.video_id!="test_7")&(ok.video_id!="test_5")]
        print(f"\n  VENUE CHECK  test_7 (2nd venue): {v7.delta:+.1%} shot error, "
              f"pose conf {v7.pose_conf:.3f}")
        if len(others):
            print(f"               other venue mean : {others.delta.abs().mean():+.1%}, "
                  f"pose conf {others.pose_conf.mean():.3f}")
        print("               -> pipeline holds across venues"
              if abs(v7.delta)<=0.15 else
              "               -> venue sensitivity; inspect test_7")

    # negative control
    t5=res[res.video_id=="test_5"]
    if len(t5):
        r=t5.iloc[0]
        n=r.get("detected",0)
        print(f"\n  CONTROL  test_5 (never trained on, no contact signal): "
              f"{n} shots vs {r.get('truth','?')} truth")
        print("           -> as expected: the pipeline does not invent shots "
              "where the input carries no signal"
              if n<=0.5*r.get("truth",1) else
              "           -> unexpectedly many; the control assumption was wrong")

if len(frames_all):
    allshots=pd.concat(frames_all)
    print("\n"+"="*78); print("CLASS MIX ACROSS ALL TEST VIDEOS"); print("="*78)
    mix=allshots.shot_class.value_counts(normalize=True)
    gtm=strokes[strokes.video_id.isin(TEST_VIDEOS)].shot_class.value_counts(normalize=True)
    for c in CLASSES:
        print(f"  {c:<9} predicted {mix.get(c,0):.3f}   truth {gtm.get(c,0):.3f}   "
              f"{mix.get(c,0)-gtm.get(c,0):+.3f}")
    print("\n  coachable shots by class:")
    for c in CLASSES:
        s=allshots[allshots.shot_class==c]
        if len(s): print(f"    {c:<9} {len(s):>4} shots, {(~s.abstain).mean():.0%} coachable")
print("="*78)

  videos run     : 5/5
  shot count err : mean +21.9%, worst +96.3%
  runtime        : 6.4x realtime (was 6.6x before the speed patch)
  coachable      : 56% of shots
  mean rally     : 6.0 shots

  within +/-10% of ground truth: 80% of videos   PASS

  VENUE CHECK  test_7 (2nd venue): +2.0% shot error, pose conf 0.803
               other venue mean : +3.7%, pose conf 0.819
               -> pipeline holds across venues

  CONTROL  test_5 (never trained on, no contact signal): 1 shots vs 27 truth
           -> as expected: the pipeline does not invent shots where the input carries no signal

CLASS MIX ACROSS ALL TEST VIDEOS
  serve     predicted 0.130   truth 0.149   -0.019
  attack    predicted 0.545   truth 0.570   -0.025
  control   predicted 0.065   truth 0.070   -0.005
  defence   predicted 0.260   truth 0.211   +0.049

  coachable shots by class:
    serve       26 shots, 100% coachable
    attack     109 shots, 98% coachable
    control     13 shots, 0% coachable
    defence   

---
## What this establishes

This is a **pipeline** test, not an accuracy test — the final models trained on 11 of these 12 videos, so shot-count agreement is expected rather than evidence of generalisation.

What it does establish: the pipeline runs unattended across multiple videos, produces stable counts, holds up in a second venue, and runs at the patched speed. Those are the properties a tool needs.

**Both Phase 2 patches are now in the code above** — sequential decoding and per-class abstention. This notebook supersedes `09_analyse.ipynb` for running the pipeline.

Next: Phase 2.5 (3D pose, for camera flexibility) or Phase 3 (coaching layer).


In [10]:
# =============================================================================
# CELL 9 — RALLY VALIDATION
#
# Rally grouping is the one component never checked against ground truth. It
# is a heuristic: a gap of more than RALLY_GAP_S between contacts starts a new
# rally. The batch run gave mean rally lengths of 8-10 shots, where real table
# tennis averages 3-5 — which suggests separate points are being merged.
#
# That matters. "Your average rally is 8 shots" is a headline coaching
# statistic, and if it is really 4, everything derived from it is wrong.
#
# `rallies.parquet` holds 282 annotated rally ENDINGS (out, net, winner,
# not_hitting_ball, double_bounce, miss_on_own_side). Each marks the last
# stroke of a point, which is exactly the boundary to validate against.
#
# No re-analysis needed — this re-groups the existing shots.parquet files.
# =============================================================================

TOL_S = 1.0      # a predicted boundary counts as matched within this window

import numpy as np, pandas as pd, json
from pathlib import Path

rallies = load("rallies") if "load" in dir() else pd.read_parquet(META/"rallies.parquet")
analysed = sorted([p.name for p in ANALYSED.iterdir()
                   if (p/"shots.parquet").exists()])
print(f"analysed videos: {analysed}")
print(f"ground-truth rally endings: {len(rallies)} across "
      f"{rallies.video_id.nunique()} videos\n")

# --- ground truth --------------------------------------------------------
gt = {}
for v in analysed:
    ends = np.sort(rallies.loc[rallies.video_id == v, "frame_120"].values)
    n_str = int((strokes.video_id == v).sum())
    gt[v] = dict(ends=ends, n_rallies=len(ends), n_strokes=n_str,
                 mean_len=n_str/max(len(ends), 1))

print("GROUND TRUTH")
print("=" * 68)
for v, g in gt.items():
    print(f"  {v:<9} {g['n_rallies']:>3} rallies, {g['n_strokes']:>3} strokes, "
          f"mean {g['mean_len']:.1f} shots/rally")
tot_r = sum(g["n_rallies"] for g in gt.values())
tot_s = sum(g["n_strokes"] for g in gt.values())
print(f"  {'TOTAL':<9} {tot_r:>3} rallies, {tot_s:>3} strokes, "
      f"mean {tot_s/max(tot_r,1):.1f}")


def regroup(frames, gap_s):
    """Frames -> rally ids, splitting where the gap exceeds gap_s."""
    rid, cur = [], 0
    for i, f in enumerate(frames):
        if i and (f - frames[i-1])/FPS > gap_s:
            cur += 1
        rid.append(cur)
    return np.array(rid)


def score(pred_ends, true_ends, tol_frames):
    """Greedy match of predicted rally boundaries to annotated ones."""
    used, tp = set(), 0
    for p in pred_ends:
        best, bd = None, tol_frames + 1
        for j, t in enumerate(true_ends):
            if j in used: continue
            d = abs(p - t)
            if d <= tol_frames and d < bd: best, bd = j, d
        if best is not None:
            used.add(best); tp += 1
    fp, fn = len(pred_ends) - tp, len(true_ends) - tp
    p = tp/max(tp+fp, 1); r = tp/max(tp+fn, 1)
    return tp, fp, fn, p, r, 2*p*r/max(p+r, 1e-9)


# --- sweep the gap parameter ---------------------------------------------
tol = int(TOL_S*FPS)
rows = []
for gap in [0.8, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0]:
    TP = FP = FN = 0; nr = 0; lens = []
    for v in analysed:
        df = pd.read_parquet(ANALYSED/v/"shots.parquet").sort_values("frame")
        if not len(df): continue
        f = df.frame.values
        rid = regroup(f, gap)
        ends = np.array([f[rid == r].max() for r in np.unique(rid)])
        nr += len(ends); lens += list(pd.Series(rid).value_counts().values)
        tp, fp, fn, *_ = score(ends, gt[v]["ends"], tol)
        TP += tp; FP += fp; FN += fn
    p = TP/max(TP+FP, 1); r = TP/max(TP+FN, 1)
    rows.append(dict(gap_s=gap, rallies=nr, mean_len=np.mean(lens) if lens else np.nan,
                     precision=p, recall=r, f1=2*p*r/max(p+r, 1e-9)))
sw = pd.DataFrame(rows)

print("\n" + "=" * 68)
print(f"RALLY-BOUNDARY F1 vs GAP THRESHOLD   (tolerance +/-{TOL_S}s)")
print("=" * 68)
print(sw.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\n  ground truth: {tot_r} rallies, mean {tot_s/max(tot_r,1):.1f} shots")

best = sw.loc[sw.f1.idxmax()]
print(f"\n  best gap: {best.gap_s}s  ->  F1 {best.f1:.3f} "
      f"(P {best.precision:.3f} / R {best.recall:.3f})")
print(f"  gives {int(best.rallies)} rallies, mean {best.mean_len:.1f} shots")
cur = sw[sw.gap_s == 2.5].iloc[0]
print(f"\n  current setting (2.5s): F1 {cur.f1:.3f}, "
      f"{int(cur.rallies)} rallies, mean {cur.mean_len:.1f}")
print(f"  change: {best.f1 - cur.f1:+.3f} F1")

# --- per-video at the best setting ---------------------------------------
print("\n" + "=" * 68)
print(f"PER VIDEO at gap={best.gap_s}s")
print("=" * 68)
pv = []
for v in analysed:
    df = pd.read_parquet(ANALYSED/v/"shots.parquet").sort_values("frame")
    if not len(df): continue
    f = df.frame.values
    rid = regroup(f, best.gap_s)
    ends = np.array([f[rid == r].max() for r in np.unique(rid)])
    tp, fp, fn, p, r, f1 = score(ends, gt[v]["ends"], tol)
    pv.append(dict(video_id=v, pred_rallies=len(ends), true_rallies=gt[v]["n_rallies"],
                   tp=tp, fp=fp, fn=fn, f1=f1,
                   pred_len=len(df)/max(len(ends), 1), true_len=gt[v]["mean_len"]))
pvd = pd.DataFrame(pv)
print(pvd.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("""
  precision low  -> splitting one point into several (gap too small)
  recall low     -> merging several points into one (gap too large)""")

# --- what limits this ----------------------------------------------------
print("\n" + "=" * 68)
print("CEILING")
print("=" * 68)
print(f"""  Rally boundaries are derived from detected contacts, so detection
  errors propagate: a missed final stroke moves the boundary, and a false
  positive in a between-point pause splits a rally.

  With detection recall around 0.88, roughly 12% of true boundaries sit on
  a stroke that was never detected. That caps boundary F1 well below 1.0
  regardless of the gap parameter — so read the number against that ceiling,
  not against perfection.""")

CAL = json.loads((META/"calibration.json").read_text())
CAL.update(rally_gap_s=float(best.gap_s), rally_boundary_f1=float(best.f1),
           rally_mean_len=float(best.mean_len))
(META/"calibration.json").write_text(json.dumps(CAL, indent=2))
print(f"\n-> saved RALLY_GAP_S = {best.gap_s}s to calibration.json")
print("   Set RALLY_GAP_S in cell 2 and re-run analyse() to apply "
      "(no re-extraction — the pose cache is reused).")

analysed videos: ['game_1', 'test_1', 'test_2', 'test_5', 'test_6', 'test_7']
ground-truth rally endings: 281 across 12 videos

GROUND TRUTH
  game_1     25 rallies, 161 strokes, mean 6.4 shots/rally
  test_1      7 rallies,  84 strokes, mean 12.0 shots/rally
  test_2      2 rallies,  29 strokes, mean 14.5 shots/rally
  test_5      7 rallies,  27 strokes, mean 3.9 shots/rally
  test_6      8 rallies,  39 strokes, mean 4.9 shots/rally
  test_7      8 rallies,  49 strokes, mean 6.1 shots/rally
  TOTAL      57 rallies, 389 strokes, mean 6.8

RALLY-BOUNDARY F1 vs GAP THRESHOLD   (tolerance +/-1.0s)
 gap_s  rallies  mean_len  precision  recall    f1
 0.800       93     3.914      0.484   0.789 0.600
 1.000       66     5.515      0.682   0.789 0.732
 1.250       62     5.871      0.726   0.789 0.756
 1.500       60     6.067      0.750   0.789 0.769
 1.750       60     6.067      0.750   0.789 0.769
 2.000       60     6.067      0.750   0.789 0.769
 2.500       59     6.169      0.746   0.